# XAI Benchmark (MURA) — Google Colab runner

**Setup (once):**
1. **Runtime → Change runtime type → GPU (T4)**.
2. Put your files in Google Drive:
   - the **MURA-v1.1** folder (or a `MURA-v1.1.zip`), and
   - the **XAI_project** code folder (or set `CODE_SRC` to a GitHub URL).
3. Run all cells and adjust the two paths in the *paths* cell.

> Tip: reading 40k small files directly from Drive is slow. For speed, upload
> `MURA-v1.1.zip` to Drive and unzip to local `/content` (see optional cell).

In [ ]:
import os, sys, glob, shutil, subprocess, pathlib
import torch
print("CUDA available:", torch.cuda.is_available())
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- paths: EDIT THESE ---
CODE_SRC   = ""                                        # optional GitHub URL for the code
DRIVE_CODE = "/content/drive/MyDrive/XAI_project"      # code folder in Drive (if not using GitHub)
MURA_ROOT  = "/content/drive/MyDrive/MURA-v1.1"        # MURA-v1.1 folder in Drive
WORK = "/content/XAI_project"

if CODE_SRC:
    if os.path.exists(WORK): shutil.rmtree(WORK)
    subprocess.run(["git", "clone", CODE_SRC, WORK], check=True)
else:
    assert os.path.exists(DRIVE_CODE), f"Set DRIVE_CODE to your code folder in Drive ({DRIVE_CODE} not found)."
    if os.path.exists(WORK): shutil.rmtree(WORK)
    shutil.copytree(DRIVE_CODE, WORK)

assert os.path.exists(os.path.join(MURA_ROOT, "train_image_paths.csv")), \
    f"Set MURA_ROOT to your MURA-v1.1 folder in Drive ({MURA_ROOT} missing train_image_paths.csv)."
print("Code:", WORK, "\nMURA:", MURA_ROOT)

In [ ]:
# --- (OPTIONAL, faster) if you uploaded MURA-v1.1.zip to Drive, unzip to local /content ---
# ZIP = "/content/drive/MyDrive/MURA-v1.1.zip"
# import zipfile
# with zipfile.ZipFile(ZIP) as z: z.extractall("/content")
# MURA_ROOT = "/content/MURA-v1.1"; print("MURA (local):", MURA_ROOT)

In [ ]:
# --- install the package (Colab already ships torch + CUDA) ---
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", WORK], check=True)

In [ ]:
# --- patch the config: dataset path + quick debug first ---
import yaml
cfg_path = os.path.join(WORK, "configs", "densenet_gradcam.yaml")
cfg = yaml.safe_load(open(cfg_path))
cfg["data"]["mura_root"] = MURA_ROOT
cfg["train"]["quick_debug"] = True    # << set to False for the full run
yaml.safe_dump(cfg, open(cfg_path, "w"), sort_keys=False)
print(open(cfg_path).read())

In [ ]:
# --- device check + run ---
sys.path.insert(0, os.path.join(WORK, "src"))
from xai_bench.utils import get_device
print("Device:", get_device())
subprocess.run([sys.executable, os.path.join(WORK, "scripts", "run_experiment.py"),
                "--config", cfg_path], check=True, cwd=WORK)

In [ ]:
# --- view results + copy back to Drive ---
import pandas as pd
print(pd.read_csv(os.path.join(WORK, "results", "results.csv")).T)
os.makedirs("/content/drive/MyDrive/XAI_results", exist_ok=True)
shutil.copy(os.path.join(WORK, "results", "results.csv"), "/content/drive/MyDrive/XAI_results/results.csv")
print("Saved results to Drive: MyDrive/XAI_results/results.csv")

### Full run
Set `cfg["train"]["quick_debug"] = False` in the config cell and re-run.
Colab free sessions can disconnect (~12h); checkpoints are saved under `WORK/checkpoints/`.